In [1]:
import pandas as pd, numpy as np
import vivarium_inputs
import gbd_mapping
import pathlib

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "India"
vehicle = "rice"

In [3]:
# Parameters
location = "india"
vehicle = "rice"


In [4]:
location = location.title()

In [5]:
pop = vivarium_inputs.get_population_structure(location).value
pop[pop > 0]

location  sex     age_start  age_end     year_start  year_end
India     Female  0.000000   0.019178    2021        2022        1.975581e+05
                  0.019178   0.076712    2021        2022        5.872100e+05
                  0.076712   0.500000    2021        2022        4.326738e+06
                  0.500000   1.000000    2021        2022        5.085507e+06
                  1.000000   2.000000    2021        2022        1.035319e+07
                  2.000000   5.000000    2021        2022        3.244181e+07
                  5.000000   10.000000   2021        2022        5.860247e+07
                  10.000000  15.000000   2021        2022        6.308955e+07
                  15.000000  20.000000   2021        2022        6.421417e+07
                  20.000000  25.000000   2021        2022        6.367111e+07
                  25.000000  30.000000   2021        2022        5.987495e+07
                  30.000000  35.000000   2021        2022        5.583203e+07
  

In [6]:
children = pop[pop.index.get_level_values("age_end") <= 5].sum()
f'{int(children):,}'

'111,336,281'

In [7]:
sim_baseline_children = pd.read_parquet(f"../../0200_pregnancy_sim/sim_results/{vehicle}/{location.lower()}/pregnancy_outcome_count.parquet")
sim_baseline_children = sim_baseline_children[
    (sim_baseline_children.scenario == 'baseline') &
    (sim_baseline_children.sub_entity == 'live_birth')
]
sim_baseline_children

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
5,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,lowest,baseline,46,0,3.0
6,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,second,baseline,46,0,7.0
7,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,middle,baseline,46,0,1.0
8,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,fourth,baseline,46,0,1.0
9,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,highest,baseline,46,0,2.0
...,...,...,...,...,...,...,...,...,...,...,...
268640,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,lowest,baseline,39,0,0.0
268641,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,second,baseline,39,0,0.0
268642,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,middle,baseline,39,0,0.0
268643,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,fourth,baseline,39,0,0.0


In [8]:
sim_baseline_children = sim_baseline_children.groupby("input_draw").value.sum().mean()
sim_baseline_children

1444153.0

In [9]:
scalar = children / sim_baseline_children
scalar

77.09451954721473

In [10]:
for result in ["ylds", "ylls", "deaths", "person_time"]:
    df = pd.read_parquet(f"../../0300_child_sim/sim_results/{vehicle}/{location.lower()}/{result}.parquet")
    df.value *= scalar
    path = pathlib.Path(f'../results/rescaled_child_results/{vehicle}/{location.lower()}/{result}.parquet')
    path.parent.mkdir(exist_ok=True, parents=True)
    df.to_parquet(path)